# CNN-FineTuning-FC 파이프라인 가이드: Grid Search · K-Fold · Before/After

동결된 `ResNet18` 백본 위에서 **FC 분류 헤드만 학습**하면서 **Norm-MultiScale · Residual/MLP 헤드 · 균형 Ridge 초기화 · 3x AugBank · CutMix/Mixup · SWA · K-Fold**를 빠르고 효율적으로 실험하실 수 있도록 구성한 통합 노트북입니다.

- Kaggle 우측 사이드바에서 **Internet: On**을 켠 뒤 위에서 아래 순서대로 실행해 주세요. (**CPU 런타임** 기준으로 바로 동작하며, GPU 사용 가능 환경에서는 자동으로 CUDA 가속이 활성화됩니다.)
- `!git pull` 이후에는 반드시 **Restart Session** 후 실행해 주세요.

In [ ]:
# 1. Split 저장소 및 프로젝트 저장소(wsn 브랜치) clone
%cd /kaggle/working
!git clone https://github.com/Doonsan-BootCamp11-4/CNN-FineTuning-Dataset-Split.git
!git clone -b wsn https://github.com/Doonsan-BootCamp11-4/CNN-FineTuning-FC.git

In [ ]:
# 2. 경로 자동 탐지 · split CSV 생성 및 무결성 점검
import subprocess, sys
from pathlib import Path
sys.path.insert(0, "/kaggle/working/CNN-FineTuning-FC")
%cd /kaggle/working/CNN-FineTuning-FC
roots = sorted(p for p in Path("/kaggle/input").glob("**/garbage_classification") if p.is_dir())
DATA_ROOT = str(roots[0]) if roots else "/kaggle/input/garbage-classification/garbage_classification"
SPLIT_CSV = "/kaggle/working/CNN-FineTuning-Dataset-Split/splits/garbage12_602020_seed42.csv"
if not Path(SPLIT_CSV).is_file():
    subprocess.run([sys.executable, "/kaggle/working/CNN-FineTuning-Dataset-Split/make_split.py", "--root", DATA_ROOT, "--out", SPLIT_CSV, "--seed", "42"], check=True)
print("DATA_ROOT:", DATA_ROOT, "| SPLIT_CSV:", SPLIT_CSV)
subprocess.run([sys.executable, "scripts/run_experiments.py", "--check-split", "--check-files", "--split-csv", SPLIT_CSV, "--data-root", DATA_ROOT], check=True)

## 3. 선언형 Grid Search (`--axis grid --shared-cache`)

[`configs/grid.py`](configs/grid.py)의 `GRID` 16조합(`Residual/MLP × CutMix/Both × AdamW/Calibrated × InvSqrt/LogitAdj`)을 일괄 탐색합니다.
*(교차검증이 필요하면 `--k-folds 5`, 실행 개수 제한이 필요하면 `--max-runs N`을 추가해 주세요. `center_power_l2`, `center256`, `aug_mode`는 `configs/grid.py`의 `SUPPORTED_OPTIONS`로 비교 가능합니다.)*

In [ ]:
# 3-1. Grid Search 실행 (Kaggle Input 등에 공용 .pt 캐시가 마운트되어 있으면 자동 연결)
from pathlib import Path
import subprocess, sys
CACHE_FILE = ""  # 직접 지정 시 경로 입력 (예: "/kaggle/input/fc-cache/fc_cache_grid.pt")
if not CACHE_FILE:
    hits = sorted(Path("/kaggle/input").glob("**/*cache*.pt")) if Path("/kaggle/input").is_dir() else []
    CACHE_FILE = str(hits[0]) if hits else ""
cmd = [
    sys.executable, "scripts/run_experiments.py", "--axis", "grid", "--shared-cache", "--co-train",
    "--split-csv", SPLIT_CSV, "--data-root", DATA_ROOT,
    "--out-dir", "/kaggle/working/CNN-FineTuning-FC/outputs_grid",
]
if CACHE_FILE:
    print("Mounted shared cache:", CACHE_FILE)
    cmd.extend(["--cache-file", CACHE_FILE])
subprocess.run(cmd, check=True)

In [ ]:
# 3-2. Grid Search 순위표 및 검증 곡선 확인
from pathlib import Path
from IPython.display import Image, Markdown, display

grid_out = Path("/kaggle/working/CNN-FineTuning-FC/outputs_grid")
display(Markdown((grid_out / "grid_summary.md").read_text(encoding="utf-8")))
grid_png = grid_out / "comparison_grid.png"
if grid_png.is_file():
    display(Image(filename=str(grid_png)))

## 4. Before vs After 및 손실·활성화 축 비교 (`--axis compare / loss / activation`)

기법 미적용 기준선(`compare_0_baseline`)부터 단계별 개선 프리셋(`--axis compare`), 손실함수 12종(`--axis loss`), 활성화함수 7종(`--axis activation`)까지 `--shared-cache --co-train` (`torch.bmm` 다중 헤드 동시 학습, 필요 시 `--auto-tune-resilient` 추가)으로 빠르게 비교할 수 있습니다.

In [ ]:
# 4-1. Before vs After 일괄 실행 (CPU 가속: --fast-cpu; 손실/활성화 비교 예시: --axis loss --shared-cache --co-train --auto-tune-resilient)
import subprocess, sys
from pathlib import Path
from IPython.display import Image, Markdown, display
subprocess.run([
    sys.executable, "scripts/run_experiments.py", "--axis", "compare", "--shared-cache", "--fast-cpu",
    "--split-csv", SPLIT_CSV, "--data-root", DATA_ROOT,
    "--out-dir", "/kaggle/working/CNN-FineTuning-FC/outputs",
], check=True)
out_root = Path("/kaggle/working/CNN-FineTuning-FC/outputs")
display(Markdown((out_root / "compare_before_after.md").read_text(encoding="utf-8")))
cmp_png = out_root / "comparison_compare.png"
if cmp_png.is_file():
    display(Image(filename=str(cmp_png)))

## 5. 최종 Test 1회 평가: Before vs Best Preset vs Top-2 Ensemble

검증 성능이 가장 높은 모델과 기준선(`compare_0_baseline`), 그리고 Top-2 확률 앙상블(`evaluate_ensemble`)을 마지막에 1회만 `test` 세트로 비교합니다. 캐시된 Test 번들을 넘기면 재추론 없이 즉시 평가됩니다.

In [ ]:
# 5-1. 기준선 vs 최고 모델 vs Top-2 앙상블 최종 Test 평가 (디스크 캐시 재사용)
from IPython.display import Image, Markdown, display
from configs import compare as compare_cfg
from fcft.data import extract_shared_feature_bundle
from fcft.evaluate import evaluate, evaluate_ensemble
from fcft.report import collect_summaries

out_dir = "/kaggle/working/CNN-FineTuning-FC/outputs"
entry_map = {e["config"]["name"]: e for e in compare_cfg.EXPERIMENTS}
ranked = collect_summaries(out_dir, names=list(entry_map.keys()))

base_cfg = {**entry_map["compare_0_baseline"]["config"], "split_csv": SPLIT_CSV, "data_root": DATA_ROOT, "out_dir": out_dir}
bundle_p = globals().get("CACHE_FILE") or None
test_bundle = extract_shared_feature_bundle(base_cfg, include_flip=True, include_descriptors=True, include_test=True, use_disk_cache=True, bundle_path=bundle_p)
m_before = evaluate(base_cfg, tta_mode="none", precomputed_test=test_bundle)

best_name = ranked[0]["name"]
best_cfg = {**entry_map[best_name]["config"], "split_csv": SPLIT_CSV, "data_root": DATA_ROOT, "out_dir": out_dir}
best_tta = entry_map[best_name].get("eval_tta_mode", "flip2")
m_after = evaluate(best_cfg, tta_mode=best_tta, precomputed_test=test_bundle)

top2_cfgs = [{**entry_map[r["name"]]["config"], "split_csv": SPLIT_CSV, "data_root": DATA_ROOT, "out_dir": out_dir} for r in ranked[:2]]
if len(top2_cfgs) >= 2:
    m_ens = evaluate_ensemble(top2_cfgs, out_name="compare_top2_ensemble", tta_mode="multiview4", temperature=1.1, precomputed_test=test_bundle)
else:
    m_ens = m_after

diff_best = (m_after["accuracy"] - m_before["accuracy"]) * 100.0
diff_ens = (m_ens["accuracy"] - m_before["accuracy"]) * 100.0
ens_label = "`compare_top2_ensemble`" if len(top2_cfgs) >= 2 else f"`{best_name}` (단일 폴백)"
summary_md = (
    "| Stage | Model | Test Acc | Macro F1 | Δ vs Before (%p) |\n"
    "| :--- | :--- | :---: | :---: | :---: |\n"
    f"| Before | `compare_0_baseline` | {m_before['accuracy']:.4f} | {m_before['macro_avg']['f1-score']:.4f} | 0.00%p |\n"
    f"| After (Best) | `{best_name}` ({best_tta}) | {m_after['accuracy']:.4f} | {m_after['macro_avg']['f1-score']:.4f} | {diff_best:+.2f}%p |\n"
    f"| Ensemble (Top-2) | {ens_label} | {m_ens['accuracy']:.4f} | {m_ens['macro_avg']['f1-score']:.4f} | {diff_ens:+.2f}%p |\n"
)
display(Markdown(summary_md))
print("Top Confusions (After Best):", m_after.get("top_confusions", [])[:5])
display(Image(filename=f"{out_dir}/{best_name}/confusion_matrix.png"))

In [ ]:
# 6. 결과 아카이브 압축
!cd /kaggle/working/CNN-FineTuning-FC && zip -rq /kaggle/working/outputs.zip outputs/ outputs_grid/
!ls -lh /kaggle/working/outputs.zip